# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR⁲ colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the FAIR⁲ Colorectal Cancer dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s in the FAIR⁲ dataset.

In [ ]:
# List all record sets by their @id
record_set_ids = [rs['@id'] for rs in metadata.recordSet] if hasattr(metadata, 'recordSet') and metadata.recordSet else []
if not record_set_ids:
    print("No record sets specified in the metadata. Attempting to infer record sets from available distributions...")
    # As a fallback, try to use distribution for record set inference
    record_set_ids = [dist['@id'] for dist in metadata.distribution] if hasattr(metadata, 'distribution') else []

if record_set_ids:
    print("Available record sets (by @id):")
    for rid in record_set_ids:
        print(f"- {rid}")
else:
    print("No record sets or distributions found in dataset metadata.")

# For each record set, print its fields and column @ids
for record_set_id in record_set_ids:
    print(f"\nRecord set: {record_set_id}")
    try:
        recordset_obj = dataset.record_set_by_id(record_set_id)
        # List the fields and columns for each record set
        field_ids = [f.get('@id', getattr(f, '@id', None)) for f in getattr(recordset_obj, 'field', [])]
        print("  Fields (@id):")
        for fid in field_ids:
            print(f"    - {fid}")
        # If columns are provided, also print their @id
        column_ids = [c.get('@id', getattr(c, '@id', None)) for c in getattr(recordset_obj, 'column', [])]
        if column_ids:
            print("  Columns (@id):")
            for cid in column_ids:
                print(f"    - {cid}")
    except Exception as e:
        print(f"  Could not load details for record set {record_set_id}: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for further analysis. All references use the `@id`s identified above.

In [ ]:
# Prepare list of record set @id's for extraction
record_sets = record_set_ids  # From the previous cell
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns in record set '{record_set_id}': {df.columns.tolist()}")
            print(df.head(2))
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Pick the first available record set for EDA
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nWill use record set: {first_record_set_id} for EDA.")
    print(f"Columns: {dataframes[first_record_set_id].columns.tolist()}")
    display(dataframes[first_record_set_id].head())
else:
    print("No dataframes loaded. Please verify the dataset schema and URLs.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by clinical criteria, normalizing numeric variables (e.g., age), and grouping by key features such as MSI status or anatomical location. All variables should be referenced by their `@id`.

In [ ]:
# --- EDA Example ---
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Use the first dataframe for demonstration
if dataframes:
    df = dataframes[first_record_set_id].copy()
    print(f"Available columns: {df.columns.tolist()}")
    
    # Guess numeric field: try to use 'age', else pick first numeric column
    possible_age_cols = [c for c in df.columns if 'age' in c.lower()]
    if possible_age_cols:
        numeric_field_id = possible_age_cols[0]
    else:
        numeric_field_id = df.select_dtypes(include=[np.number]).columns[0] if not df.select_dtypes(include=[np.number]).empty else None

    if numeric_field_id:
        print(f"Using numeric field for EDA: {numeric_field_id}")

        # Use a threshold for filtering - median by default
        threshold = df[numeric_field_id].median() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head(3))

        # Normalization
        mu = filtered_df[numeric_field_id].mean()
        sigma = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mu) / sigma
        print(f"\nNormalized {numeric_field_id} (mean={mu:.2f}, std={sigma:.2f}):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical field (e.g., 'msi', 'status', or anatomical)
        cat_fields = [col for col in df.columns if any(k in col.lower() for k in ["msi", "status", "anatomical", "site", "sex", "gender"])]
        group_field_id = cat_fields[0] if cat_fields else df.select_dtypes(include=[object]).columns[0] if df.select_dtypes(include=[object]).shape[1] else None
        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            display(grouped)
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize distributions or relationships between key fields.

Below is an example of age distribution (or the selected numeric field) by MSI status (or another group):

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    if filtered_df[group_field_id].nunique() > 1:
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.show()
    else:
        sns.histplot(filtered_df[numeric_field_id], kde=True)
        plt.xlabel(f"{numeric_field_id}")
        plt.title(f"Distribution of {numeric_field_id}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook demonstrated loading, basic EDA, and initial visualization of the FAIR⁲ second primary colorectal cancer dataset using the `mlcroissant` library. Further analysis may involve prognosis modeling, biological subtyping, or advanced stratified analyses using the record set and field `@id`s as references throughout.